In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Inicializace Qubitů

<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut pomocí následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Když je Circuit spuštěn na kvantové procesorové jednotce (QPU) od IBM&reg;, je na začátek Circuit obvykle vložen implicitní reset, aby bylo zajištěno, že Qubity jsou inicializovány na nulu. To je řízeno příznakem `init_qubits`, nastaveným jako [možnost spuštění primitivy](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Nedokonalosti v procesu resetu však mohou způsobovat chyby při přípravě stavu. Aby se tato chyba zmírnila, QPU také vkládá dobu prodlevy opakování (neboli `rep_delay`) mezi Circuits. Každý Backend má jiný výchozí `rep_delay`, ale obvykle je nastaven tak, aby vyvažoval věrnost resetu a celkovou dobu spouštění. Výchozí `rep_delay` pro konkrétní QPU zjistíš spuštěním `backend.default_rep_delay`.

Protože všechny IBM QPU používají dynamické spouštění s opakovací frekvencí, můžeš změnit `rep_delay` pro každý job. Circuits, které odesíláš v rámci primitivního jobu, jsou pro spuštění na QPU sdruženy dohromady. Tyto Circuits jsou spouštěny iterací přes Circuits pro každý požadovaný shot; spouštění probíhá sloupcově přes matici Circuits a shotů, jak je znázorněno na následujícím obrázku.

![První sloupec představuje shot 0. Circuits jsou spouštěny v pořadí od 0 do 3. Druhý sloupec představuje shot 1. Circuits jsou spouštěny v pořadí od 0 do 3. Zbývající sloupce se řídí stejným vzorem. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matice sloupcového spouštění")

Protože `rep_delay` je vkládán mezi Circuits, každý shot spouštění narazí na tuto prodlevu. Jak tedy `rep_delay` snižuješ, celková doba spouštění na QPU se zkracuje, za cenu rostoucí míry chyb při přípravě stavu, jak ilustruje následující obrázek:

![Tento obrázek ukazuje, že jak se hodnota `rep_delay` snižuje, míra chyb při přípravě stavu roste.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Prodleva opakování versus míra chyb")

Pokud nastavíš `rep_delay=0` i `init_qubits=False`, Circuits se „sloučí" dohromady, protože Qubity začnou v konečném stavu z předchozího shotu.

Všimni si, že ačkoli jsou Circuits v rámci primitivního jobu sdruženy pro spouštění na QPU, pořadí, ve kterém jsou Circuits z PUBů spouštěny, není zaručeno. Například pokud odešleš `pubs=[pub1, pub2]`, Circuits z `pub1` nemusí proběhnout před těmi z `pub2`. Rovněž není zaručeno, že Circuits ze stejného jobu poběží jako jediná dávka na QPU.

## Zadání `rep_delay` pro primitivní job
> **Note:** Vždy ověř podporovaný rozsah `rep_delay` pro konkrétní QPU, kterou používáš. Tyto hodnoty nejsou pro každou QPU stejné a mohou se v průběhu času měnit.
> 
> Upozorňujeme, že zvýšení `rep_delay` bude mít přímý dopad na dobu spouštění a spotřebu kapacity.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Další kroky
> **Tip:** - Vyzkoušej příklad v tutoriálu [Kvantový algoritmus přibližné optimalizace (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Přečti si, jak [začít s Estimatorem](/guides/get-started-with-estimator).
>   - Přečti si, jak [začít se Samplerem](/guides/get-started-with-sampler).